In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv(dotenv_path='../.env')

True

In [3]:
model = ChatOpenAI(model='gpt-4o-mini')

In [6]:
# create a state graph
class LLMstate(TypedDict):
    question: str
    answer: str

In [5]:
def llm_qa(state: LLMstate) -> LLMstate:
    #extract question from state
    question = state['question']

    # form a prompt
    prompt = f"Answer the following question: {question}"
    # ask to llm
    response = model.invoke(question)
    
    # update answer in state
    state['answer'] = response.content
    return state

In [9]:
# Create a graph

graph = StateGraph(LLMstate)

# Add nodes
graph.add_node('llm_qa', llm_qa)

# Add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# Create a workflow
# workflow = graph.to_workflow()
workflow = graph.compile()



In [11]:
# Execute the workflow
initial_state = LLMstate(question="What is the capital of France?", answer="")
final_state = workflow.invoke(initial_state)

final_state


{'question': 'What is the capital of France?',
 'answer': 'The capital of France is Paris.'}